# 39. 평가 자동화 — 무엇을 어떻게 측정할까

> **제39장** · **이론편 대응: 21.5절(RAG 평가), 25.5절(운영 지표)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음 (27장의 sentence-transformers 사용)
> **API 키**: 선택 (LLM-as-Judge에 사용)

---

## 이 장에서 하는 일

여러 장에서 "측정해야 한다"고 말했다. **그런데 무엇을 어떻게 측정하나.**

| 장 | 남긴 숙제 |
|---|---|
| 33장 | "퍼플렉서티만으로 판단하지 말 것" |
| 32장 | "품질 측정이 가장 어렵다" |
| 28장 | "답변이 맞는지 어떻게 아나" |

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | **문자열 지표의 한계** ★ | 21.5절 |
| 2 | 의미 기반 평가 | 21.5절 |
| 3 | **LLM-as-Judge** ★ | 25.5절 |
| 4 | 평가자 일치도 | 25.5절 |
| 5 | RAG 전용 지표 | 21.5절 |
| 6 | 평가 데이터 만들기 | 21.5절 |
| 7 | 회귀 시험 | 25.5절 |
| 8 | 평가 파이프라인 | 25.5절 |

**1절과 3절이 핵심이다.** 왜 단순 비교가 안 되는지 보이고,
그 대안이 어떤 특성을 갖는지 확인한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import json
import re
import os
import time
from collections import Counter
from pathlib import Path

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)

# API 키 (3절에 선택적으로 사용)
root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
try:
    from dotenv import load_dotenv
    load_dotenv(root / ".env")
except ImportError:
    pass

API_KEY, BASE_URL, MODEL = None, None, "gpt-4o-mini"
for env_name, base, model in [
        ("OPENAI_API_KEY", None, "gpt-4o-mini"),
        ("GROQ_API_KEY", "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
        ("GEMINI_API_KEY", "https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.0-flash")]:
    if os.getenv(env_name):
        API_KEY, BASE_URL, MODEL = os.getenv(env_name), base, model
        break

print(f"API 키: {'있음 (' + MODEL + ')' if API_KEY else '없음 — 3절은 구조만 확인'}")

---

## 1. 문자열 지표의 한계 ★ — 이론편 21.5절

**가장 단순한 방법부터 본다.** 예측과 정답을 문자열로 비교하는 것이다.

| 지표 | 계산 | 쓰이는 곳 |
|---|---|---|
| **EM** (Exact Match) | 완전히 같은가 | 단답형 QA |
| **F1** | 겹치는 토큰 비율 | 추출형 QA |
| **ROUGE** | n-gram 겹침 | 요약 |
| **BLEU** | n-gram 정밀도 | 번역 |

**문제가 있다.** 직접 계산해 보면 안다.

In [ ]:
from collections import Counter


def exact_match(pred, gold):
    """완전 일치 — 전처리 후 비교"""
    return pred.strip().lower() == gold.strip().lower()


def token_f1(pred, gold):
    """토큰 단위 F1 (이론편 9.3절의 F1 과 같은 개념)"""
    p_tokens = pred.lower().split()
    g_tokens = gold.lower().split()

    if not p_tokens or not g_tokens:
        return 0.0

    common = Counter(p_tokens) & Counter(g_tokens)
    n_common = sum(common.values())

    if n_common == 0:
        return 0.0

    precision = n_common / len(p_tokens)
    recall = n_common / len(g_tokens)
    return 2 * precision * recall / (precision + recall)


print("=" * 78)
print("문자열 지표의 문제")
print("=" * 78)
print()
print("질문: '대한민국의 수도는?'  정답: '서울'")
print()
print(f"{'모델 답변':<28}{'EM':<10}{'F1':<12}{'사람이 보면'}")
print("-" * 78)

cases = [
    ("서울",                  "정답"),
    ("서울입니다",             "정답"),
    ("정답은 서울입니다",       "정답"),
    ("서울특별시",             "정답"),
    ("대한민국의 수도는 서울",   "정답"),
    ("부산",                  "오답"),
]

gold = "서울"
for pred, human in cases:
    em = exact_match(pred, gold)
    f1 = token_f1(pred, gold)
    mark = "  ←" if (human == "정답" and not em) else ""
    print(f"{pred:<28}{str(em):<10}{f1:<12.4f}{human}{mark}")

print("-" * 78)
print()
print("[문제가 보인다]")
print("  '서울입니다' 는 명백히 정답인데 EM 도 F1 도 0 이다.")
print("  조사 하나, 어미 하나 때문에 오답으로 처리된다.")
print()
print("  한국어는 특히 심하다 — 조사·어미 변화가 많기 때문이다.")

In [ ]:
from collections import Counter


def get_ngrams(text, n):
    tokens = text.lower().split()
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def rouge_n(pred, gold, n=1):
    """ROUGE-N — 정답의 n-gram 중 예측에 있는 비율 (재현율 중심)"""
    p_grams = get_ngrams(pred, n)
    g_grams = get_ngrams(gold, n)
    if not g_grams:
        return 0.0
    overlap = sum((p_grams & g_grams).values())
    return overlap / sum(g_grams.values())


def rouge_l(pred, gold):
    """ROUGE-L — 최장 공통 부분수열 기반"""
    p = pred.lower().split()
    g = gold.lower().split()
    if not p or not g:
        return 0.0

    # LCS 길이 (동적 계획법)
    dp = [[0] * (len(g) + 1) for _ in range(len(p) + 1)]
    for i in range(1, len(p) + 1):
        for j in range(1, len(g) + 1):
            if p[i-1] == g[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    lcs = dp[len(p)][len(g)]
    precision = lcs / len(p)
    recall = lcs / len(g)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


print("=" * 78)
print("ROUGE — 요약 평가에 쓰인다")
print("=" * 78)
print()

summary_cases = [
    ("고양이가 매트 위에 앉아 있다", "고양이가 매트 위에 앉았다", "거의 같음"),
    ("매트 위에 고양이가 있다", "고양이가 매트 위에 앉았다", "같은 뜻"),
    ("동물이 바닥에 있다", "고양이가 매트 위에 앉았다", "비슷한 뜻"),
    ("날씨가 좋다", "고양이가 매트 위에 앉았다", "무관"),
]

print(f"{'예측':<28}{'ROUGE-1':<12}{'ROUGE-2':<12}{'ROUGE-L':<12}{'사람 판단'}")
print("-" * 78)
for pred, ref, human in summary_cases:
    print(f"{pred:<28}{rouge_n(pred, ref, 1):<12.4f}"
          f"{rouge_n(pred, ref, 2):<12.4f}{rouge_l(pred, ref):<12.4f}{human}")
print("-" * 78)
print()
print("[관찰]")
print("  '매트 위에 고양이가 있다' 는 같은 뜻인데 ROUGE-2 가 낮다.")
print("  단어 순서가 바뀌었기 때문이다.")
print()
print("[문자열 지표의 근본 한계]")
print("  **표현이 다르면 뜻이 같아도 낮게 나온다**")
print("  → 2절의 의미 기반 평가가 필요한 이유")

### 그렇다면 문자열 지표는 쓸모없는가

**아니다.** 적합한 곳이 있다.

| 상황 | 적합한 지표 |
|---|---|
| 정답이 짧고 명확 (숫자, 이름) | **EM** |
| 추출형 QA (원문에서 가져옴) | **F1** |
| 요약 (참조 요약이 있음) | **ROUGE** |
| 번역 (참조 번역이 있음) | **BLEU** |
| 자유 형식 답변 | **부적합** |

**계산이 빠르고 재현 가능하다**는 장점이 크다.
문제는 **자유 형식 답변에 쓸 때**다.

---

## 2. 의미 기반 평가 — 이론편 21.5절

**27장의 임베딩을 평가에 쓴다.**

예측과 정답을 벡터로 바꿔 코사인 유사도를 재면, 표현이 달라도 뜻이 같으면 높게 나온다.

In [ ]:
import numpy as np
import time
from sentence_transformers import SentenceTransformer

print("의미 평가용 모델 로드 중... (27장에서 받았다면 즉시)")
t0 = time.time()
evaluator = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print(f"완료 {time.time()-t0:.1f}초")
print()


def semantic_similarity(pred, gold):
    """의미 유사도 (27장 2절)"""
    emb = evaluator.encode([pred, gold], normalize_embeddings=True)
    return float(emb[0] @ emb[1])


print("=" * 78)
print("문자열 지표 vs 의미 유사도")
print("=" * 78)
print()
print("질문: '대한민국의 수도는?'  정답: '서울'")
print()
print(f"{'모델 답변':<28}{'EM':<10}{'F1':<12}{'의미 유사도':<14}{'사람 판단'}")
print("-" * 78)

for pred, human in cases:
    em = exact_match(pred, gold)
    f1 = token_f1(pred, gold)
    sem = semantic_similarity(pred, gold)
    mark = "  ←" if (human == "정답" and not em and sem > 0.7) else ""
    print(f"{pred:<28}{str(em):<10}{f1:<12.4f}{sem:<14.4f}{human}{mark}")

print("-" * 78)
print()
print("[개선된 점]")
print("  '서울입니다' 가 높은 유사도를 받는다.")
print("  표현 차이를 넘어 의미를 본다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("의미 유사도의 함정")
print("=" * 78)
print()

tricky = [
    ("서울", "서울", "완전 일치"),
    ("부산", "서울", "**오답인데 유사도가 높다**"),
    ("도쿄", "서울", "오답 (다른 나라)"),
    ("수도가 아닙니다", "서울", "오답"),
    ("서울이 아닙니다", "서울", "**부정문 — 정반대인데 유사**"),
]

print(f"{'예측':<24}{'정답':<12}{'의미 유사도':<16}{'판단'}")
print("-" * 78)
sims = []
for pred, g, note in tricky:
    sim = semantic_similarity(pred, g)
    sims.append(sim)
    print(f"{pred:<24}{g:<12}{sim:<16.4f}{note}")
print("-" * 78)
print()
print("[치명적 문제]")
print("  '부산' 과 '서울' 은 둘 다 도시라서 유사도가 높다.")
print("  '서울이 아닙니다' 는 정반대 뜻인데 단어가 겹쳐 높게 나온다.")
print()
print("  → 의미 유사도는 **오답을 걸러내지 못한다**")
print()

fig, ax = plt.subplots(figsize=(9, 4))
labels = [t[0] for t in tricky]
colors = ["#0D9488" if i == 0 else "#DC2626" for i in range(len(tricky))]
bars = ax.barh(range(len(labels)), sims, color=colors)
for b, s in zip(bars, sims):
    ax.text(s + 0.01, b.get_y() + b.get_height()/2, f"{s:.3f}",
            va="center", fontsize=9)
ax.axvline(0.8, color="#1E40AF", linestyle="--", linewidth=1.5)
ax.text(0.81, len(labels) - 0.5, "임계값 0.8?", fontsize=8, color="#1E40AF")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("의미 유사도")
ax.set_xlim(0, 1.1)
ax.set_title("초록만 정답 — 임계값으로 나눌 수 있나")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print("[결론]")
print("  의미 유사도는 문자열 지표보다 낫지만 **정답 판정에는 부족하다**.")
print("  → 3절의 LLM-as-Judge 가 필요한 이유")

---

## 3. LLM-as-Judge ★ — 이론편 25.5절

**LLM에게 채점을 시킨다.**

사람이 하던 판단을 모델에게 맡기는 것이다. 사람보다 싸고 빠르며,
문자열·의미 지표가 못 잡는 것을 잡는다.

```
[채점 프롬프트]
질문: ...
정답: ...
모델 답변: ...

이 답변이 정답과 같은 내용인가요? 0 또는 1로만 답하세요.
```

**주의할 점이 많다.** 5절에서 다룬다.

In [ ]:
import json
import re


def call_llm(messages, max_tokens=200, temperature=0.0):
    """LLM 호출 (25장 방식)"""
    if not API_KEY:
        return None
    try:
        from openai import OpenAI
        kwargs = {"api_key": API_KEY}
        if BASE_URL:
            kwargs["base_url"] = BASE_URL
        client = OpenAI(**kwargs)
        r = client.chat.completions.create(
            model=MODEL, messages=messages,
            max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    except Exception as e:
        print(f"[오류] {type(e).__name__}: {str(e)[:100]}")
        return None


JUDGE_BINARY = """당신은 답변을 채점하는 평가자입니다.

주어진 질문에 대해 모델의 답변이 정답과 같은 내용인지 판단하세요.

판단 기준:
- 표현이 달라도 내용이 같으면 정답으로 봅니다.
- 부분적으로만 맞으면 오답으로 봅니다.
- 정답에 없는 내용을 덧붙였더라도 핵심이 맞으면 정답입니다.

반드시 아래 JSON 형식으로만 답하세요.
{"score": 0 또는 1, "reason": "판단 근거 한 문장"}"""


def judge_binary(question, prediction, reference):
    """이진 채점 — 맞았나 틀렸나"""
    user = (f"[질문]\n{question}\n\n"
            f"[정답]\n{reference}\n\n"
            f"[모델 답변]\n{prediction}")
    result = call_llm([
        {"role": "system", "content": JUDGE_BINARY},
        {"role": "user", "content": user},
    ], max_tokens=150)

    if result is None:
        return None, None

    cleaned = result.strip().replace("```json", "").replace("```", "")
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start:end+1]
    try:
        data = json.loads(cleaned)
        return int(data.get("score", 0)), data.get("reason", "")
    except json.JSONDecodeError:
        m = re.search(r"[01]", cleaned)
        return (int(m.group()) if m else None), cleaned[:60]


print("=" * 78)
print("LLM-as-Judge — 이진 채점")
print("=" * 78)

question = "대한민국의 수도는?"
test_answers = [pred for pred, _ in cases] + ["서울이 아닙니다"]
human_labels = [1, 1, 1, 1, 1, 0, 0]      # 사람의 판단

if API_KEY:
    print(f"\n{'모델 답변':<26}{'EM':<8}{'의미':<10}{'LLM':<8}{'사람':<8}{'근거'}")
    print("-" * 78)
    judge_scores = []
    for pred, human in zip(test_answers, human_labels):
        em = int(exact_match(pred, gold))
        sem = semantic_similarity(pred, gold)
        score, reason = judge_binary(question, pred, gold)
        judge_scores.append(score)
        mark = "" if score == human else "  ←불일치"
        print(f"{pred:<26}{em:<8}{sem:<10.3f}{str(score):<8}{human:<8}"
              f"{str(reason)[:22]}{mark}")
    print("-" * 78)

    agree = sum(1 for s, h in zip(judge_scores, human_labels) if s == h)
    print(f"사람과 일치: {agree}/{len(human_labels)} ({agree/len(human_labels)*100:.0f}%)")
else:
    print()
    print("(API 키가 없어 실제 채점은 건너뜁니다)")
    print()
    print("채점 프롬프트 구조")
    print("-" * 78)
    print(JUDGE_BINARY)
    print("-" * 78)
    print()
    print("[기대하는 동작]")
    print("  '서울입니다'    → 1 (표현만 다름)")
    print("  '부산'          → 0 (다른 도시)")
    print("  '서울이 아닙니다' → 0 (부정문)")
    print()
    print("  의미 유사도가 못 잡던 마지막 두 경우를 LLM 은 잡는다.")

In [ ]:
JUDGE_RUBRIC = """당신은 답변을 채점하는 평가자입니다.

아래 기준으로 1~5점을 매기세요.

5점: 정확하고 완전하며 명확함
4점: 정확하지만 일부 세부가 부족함
3점: 대체로 맞지만 중요한 내용이 빠짐
2점: 부분적으로만 맞음
1점: 틀렸거나 관련 없음

반드시 아래 JSON 형식으로만 답하세요.
{"score": 1~5, "reason": "판단 근거"}"""


def judge_rubric(question, prediction, reference=None):
    """루브릭 채점 — 1~5점"""
    user = f"[질문]\n{question}\n"
    if reference:
        user += f"\n[참고 답변]\n{reference}\n"
    user += f"\n[평가할 답변]\n{prediction}"

    result = call_llm([
        {"role": "system", "content": JUDGE_RUBRIC},
        {"role": "user", "content": user},
    ], max_tokens=200)

    if result is None:
        return None, None

    cleaned = result.strip().replace("```json", "").replace("```", "")
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start:end+1]
    try:
        data = json.loads(cleaned)
        return int(data.get("score", 0)), data.get("reason", "")
    except json.JSONDecodeError:
        m = re.search(r"[1-5]", cleaned)
        return (int(m.group()) if m else None), cleaned[:60]


print("=" * 78)
print("채점 방식 세 가지")
print("=" * 78)
print()
print(f"{'방식':<20}{'출력':<20}{'장점':<22}{'단점'}")
print("-" * 78)
print(f"{'이진 (0/1)':<20}{'맞음/틀림':<20}{'단순, 일관됨':<22}{'정도를 알 수 없음'}")
print(f"{'루브릭 (1~5)':<20}{'점수':<20}{'세밀함':<22}{'기준이 흔들림'}")
print(f"{'쌍대 비교':<20}{'A/B 중 선택':<20}{'가장 일관됨':<22}{'절대 수준 모름'}")
print("-" * 78)
print()

if API_KEY:
    print("루브릭 채점 예시")
    print(f"\n{'답변':<40}{'점수':<10}{'근거'}")
    print("-" * 78)
    rubric_cases = [
        "서울입니다.",
        "서울입니다. 조선시대부터 수도였으며 인구는 약 950만 명입니다.",
        "아마 서울일 것 같습니다.",
        "한국에 있는 도시입니다.",
        "부산입니다.",
    ]
    for ans in rubric_cases:
        score, reason = judge_rubric(question, ans, gold)
        print(f"{ans[:38]:<40}{str(score):<10}{str(reason)[:26]}")
    print("-" * 78)
else:
    print("(API 키 없음 — 루브릭 프롬프트 구조)")
    print("-" * 78)
    print(JUDGE_RUBRIC)
    print("-" * 78)

### LLM-as-Judge의 알려진 편향

**모델의 판단에도 치우침이 있다.** 알고 써야 한다.

| 편향 | 내용 | 대응 |
|---|---|---|
| **위치 편향** | 먼저 제시된 답을 선호 | 순서를 바꿔 두 번 평가 |
| **길이 편향** | 긴 답을 더 좋게 평가 | 길이를 기준에 명시 |
| **자기 선호** | 같은 모델이 만든 답을 선호 | 다른 모델로 채점 |
| **관대함** | 대체로 후하게 줌 | 기준을 엄격히 서술 |
| **형식 선호** | 목록·굵은 글씨를 선호 | 형식과 내용을 분리 |

In [ ]:
print("=" * 78)
print("위치 편향 확인 — 쌍대 비교에서")
print("=" * 78)
print()

JUDGE_PAIRWISE = """두 답변 중 어느 것이 질문에 더 잘 답했는지 고르세요.

판단 기준: 정확성 > 완결성 > 명확성

반드시 아래 JSON 형식으로만 답하세요.
{"winner": "A" 또는 "B" 또는 "tie", "reason": "근거"}"""


def judge_pairwise(question, answer_a, answer_b):
    """쌍대 비교"""
    user = (f"[질문]\n{question}\n\n"
            f"[답변 A]\n{answer_a}\n\n"
            f"[답변 B]\n{answer_b}")
    result = call_llm([
        {"role": "system", "content": JUDGE_PAIRWISE},
        {"role": "user", "content": user},
    ], max_tokens=150)

    if result is None:
        return None
    cleaned = result.strip().replace("```json", "").replace("```", "")
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start:end+1]
    try:
        return json.loads(cleaned).get("winner", "?")
    except json.JSONDecodeError:
        return cleaned[:20]


ans_short = "서울입니다."
ans_long = "대한민국의 수도는 서울특별시입니다. 1394년 조선 건국 이후 수도로 기능해 왔습니다."

if API_KEY:
    print("같은 두 답변을 순서만 바꿔 평가한다")
    print()
    w1 = judge_pairwise(question, ans_short, ans_long)
    w2 = judge_pairwise(question, ans_long, ans_short)

    print(f"  1차: A=짧은 답, B=긴 답  → 승자 {w1}")
    print(f"  2차: A=긴 답, B=짧은 답  → 승자 {w2}")
    print()

    # 일관성 확인
    consistent = (w1 == "A" and w2 == "B") or (w1 == "B" and w2 == "A") or \
                 (w1 == "tie" and w2 == "tie")
    if consistent:
        print("  [일관됨] 순서를 바꿔도 같은 답변을 선택했다")
    else:
        print("  [위치 편향] 순서에 따라 판단이 달라졌다")
        print("  → 실무에서는 항상 양방향으로 평가해 평균낸다")
else:
    print("(API 키 없음 — 검사 방법만 확인)")
    print()
    print("  1) 같은 두 답변을 A/B 순서로 평가")
    print("  2) 순서를 바꿔 B/A 로 다시 평가")
    print("  3) 결과가 뒤집히지 않으면 일관된 것")
    print()
    print("  뒤집히면 위치 편향이 있다는 뜻이다.")

print()
print("=" * 78)
print("편향을 줄이는 실무 방법")
print("=" * 78)
print()
print("  1) 양방향 평가 후 평균 — 위치 편향 상쇄")
print("  2) 여러 번 평가 후 다수결 — 35장의 Self-Consistency")
print("  3) 채점 모델과 평가 대상 모델을 다르게")
print("  4) 기준을 구체적으로 서술 — '좋은 답변'이 아니라 항목별로")
print("  5) 소수 표본은 사람이 검증 — 4절")

---

## 4. 평가자 일치도 — 이론편 25.5절

**LLM 채점을 믿어도 되나?**

사람의 판단과 얼마나 일치하는지 재야 한다.
단순 일치율만으로는 부족하고, **우연히 맞을 확률**을 빼야 한다.

In [ ]:
import numpy as np
from collections import Counter


def cohens_kappa(rater_a, rater_b):
    """코헨의 카파 — 우연 일치를 보정한 일치도

    kappa = (관측 일치율 - 기대 일치율) / (1 - 기대 일치율)
    """
    n = len(rater_a)
    observed = sum(1 for a, b in zip(rater_a, rater_b) if a == b) / n

    count_a, count_b = Counter(rater_a), Counter(rater_b)
    labels = set(rater_a) | set(rater_b)
    expected = sum((count_a[l] / n) * (count_b[l] / n) for l in labels)

    if expected >= 1.0:
        return 1.0
    return (observed - expected) / (1 - expected)


print("=" * 78)
print("왜 단순 일치율로는 부족한가")
print("=" * 78)
print()
print("극단적인 예: 두 평가자가 모두 90%를 '정답'이라 찍는다면")
print()

n = 100
np.random.seed(42)
# 무작위로 90% 를 1로
lazy_a = np.random.choice([1, 0], n, p=[0.9, 0.1]).tolist()
lazy_b = np.random.choice([1, 0], n, p=[0.9, 0.1]).tolist()

obs = sum(1 for a, b in zip(lazy_a, lazy_b) if a == b) / n
kap = cohens_kappa(lazy_a, lazy_b)

print(f"  단순 일치율: {obs:.4f}   ← 높아 보인다")
print(f"  카파       : {kap:.4f}   ← 실제로는 우연 수준")
print()
print("  둘 다 그냥 '정답'을 많이 찍었을 뿐인데 일치율은 높다.")
print("  카파는 이런 착시를 걷어낸다.")
print()

print("=" * 78)
print("카파 해석 기준")
print("=" * 78)
print(f"{'카파 값':<20}{'해석'}")
print("-" * 78)
for rng, label in [("< 0.00", "일치 없음"), ("0.00 ~ 0.20", "미미함"),
                   ("0.21 ~ 0.40", "약함"), ("0.41 ~ 0.60", "보통"),
                   ("0.61 ~ 0.80", "상당함"), ("0.81 ~ 1.00", "거의 완전")]:
    print(f"{rng:<20}{label}")
print("-" * 78)
print()

# 실제 예시
rater_human = [1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1]
rater_llm   = [1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1]

obs2 = sum(1 for a, b in zip(rater_human, rater_llm) if a == b) / len(rater_human)
kap2 = cohens_kappa(rater_human, rater_llm)

print("예시: 사람 평가 vs LLM 평가 (15개 표본)")
print(f"  사람: {rater_human}")
print(f"  LLM : {rater_llm}")
print()
print(f"  단순 일치율: {obs2:.4f}")
print(f"  카파       : {kap2:.4f}  → ", end="")
if kap2 > 0.8:
    print("거의 완전")
elif kap2 > 0.6:
    print("상당함 — 실무에서 쓸 만함")
elif kap2 > 0.4:
    print("보통 — 주의 필요")
else:
    print("약함 — 채점 기준을 재검토해야")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("일치도가 낮을 때 무엇을 볼까")
print("=" * 78)
print()

# 불일치 사례 분석
disagree = [(i, h, l) for i, (h, l) in
            enumerate(zip(rater_human, rater_llm)) if h != l]

print(f"불일치 사례: {len(disagree)}건 / {len(rater_human)}건")
print()
print(f"{'번호':<10}{'사람':<10}{'LLM':<10}{'유형'}")
print("-" * 78)
for i, h, l in disagree:
    kind = "LLM이 관대함" if l > h else "LLM이 엄격함"
    print(f"{i:<10}{h:<10}{l:<10}{kind}")
print("-" * 78)
print()

n_lenient = sum(1 for _, h, l in disagree if l > h)
n_strict = sum(1 for _, h, l in disagree if l < h)
print(f"LLM 이 관대한 경우: {n_lenient}건")
print(f"LLM 이 엄격한 경우: {n_strict}건")
print()
if n_lenient > n_strict:
    print("  → LLM 이 대체로 후하다. 채점 기준을 더 엄격히 서술해야 한다.")
elif n_strict > n_lenient:
    print("  → LLM 이 대체로 깐깐하다. 허용 범위를 명시해야 한다.")
else:
    print("  → 한쪽으로 치우치지 않았다. 개별 사례를 살펴봐야 한다.")

fig, ax = plt.subplots(figsize=(8, 4))
categories = ["일치", "LLM 관대", "LLM 엄격"]
values = [len(rater_human) - len(disagree), n_lenient, n_strict]
colors_a = ["#0D9488", "#EA580C", "#DC2626"]
bars = ax.bar(categories, values, color=colors_a)
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.2, str(v),
            ha="center", fontsize=11)
ax.set_ylabel("건수")
ax.set_title(f"사람 vs LLM 평가 (카파 {kap2:.3f})")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("[실무 절차]")
print("  1) 표본 50~100건을 사람이 채점한다")
print("  2) 같은 표본을 LLM 이 채점한다")
print("  3) 카파를 계산한다")
print("  4) 0.6 이상이면 LLM 채점을 신뢰하고 전체에 적용한다")
print("  5) 낮으면 프롬프트를 고쳐 다시 측정한다")

---

## 5. RAG 전용 지표 — 이론편 21.5절

**RAG는 두 단계이므로 각각 평가해야 한다.**

| 단계 | 지표 | 27~28장에서 |
|---|---|---|
| 검색 | Recall@k, MRR | 27장 8절에서 다룸 |
| **생성** | 충실도, 관련성 | 여기서 다룸 |

**충실도(faithfulness)**: 답변이 검색된 문서에 근거하는가
**관련성(relevance)**: 답변이 질문에 답하는가

In [ ]:
print("=" * 78)
print("RAG 평가의 세 축")
print("=" * 78)
print()
print(f"{'지표':<20}{'질문':<34}{'무엇을 잡나'}")
print("-" * 78)
print(f"{'검색 재현율':<20}{'정답 문서를 찾았나':<34}{'검색 실패'}")
print(f"{'충실도':<20}{'문서에 근거한 답인가':<34}{'환각'}")
print(f"{'답변 관련성':<20}{'질문에 답했나':<34}{'동문서답'}")
print("-" * 78)
print()

FAITHFULNESS_PROMPT = """당신은 답변의 근거를 검증하는 평가자입니다.

주어진 답변이 제공된 문서에만 근거하는지 판단하세요.

판단 기준:
- 문서에 없는 사실을 말하면 낮은 점수
- 문서 내용을 정확히 반영하면 높은 점수
- 문서를 요약하거나 재구성한 것은 괜찮음

반드시 아래 JSON 형식으로만 답하세요.
{"score": 0.0~1.0, "unsupported": ["문서에 없는 내용"], "reason": "근거"}"""


def judge_faithfulness(answer, context):
    """충실도 평가 — 답변이 문서에 근거하는가"""
    user = f"[제공된 문서]\n{context}\n\n[답변]\n{answer}"
    result = call_llm([
        {"role": "system", "content": FAITHFULNESS_PROMPT},
        {"role": "user", "content": user},
    ], max_tokens=250)

    if result is None:
        return None, None
    cleaned = result.strip().replace("```json", "").replace("```", "")
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start:end+1]
    try:
        data = json.loads(cleaned)
        return float(data.get("score", 0)), data.get("unsupported", [])
    except (json.JSONDecodeError, ValueError):
        return None, cleaned[:60]


# 28장의 사내 규정 예제
context_doc = "재택근무는 주 2회까지 신청 가능하며 팀장 승인이 필요합니다. 신청은 최소 3일 전에 합니다."

answers_to_check = [
    ("재택근무는 주 2회까지 가능하며 팀장 승인이 필요합니다.", "충실함"),
    ("재택근무는 주 2회까지 가능합니다. 통신비도 지원됩니다.", "**환각 포함**"),
    ("재택근무는 주 3회까지 가능합니다.", "**사실 왜곡**"),
    ("연차는 15일 부여됩니다.", "**무관한 답변**"),
]

print("충실도 평가 예시")
print(f"제공 문서: {context_doc}")
print()

if API_KEY:
    print(f"{'답변':<44}{'충실도':<12}{'문서에 없는 내용'}")
    print("-" * 78)
    for ans, note in answers_to_check:
        score, unsupported = judge_faithfulness(ans, context_doc)
        unsup_str = str(unsupported)[:24] if unsupported else "—"
        print(f"{ans[:42]:<44}{str(score):<12}{unsup_str}")
    print("-" * 78)
else:
    print(f"{'답변':<50}{'기대 판정'}")
    print("-" * 78)
    for ans, note in answers_to_check:
        print(f"{ans[:48]:<50}{note}")
    print("-" * 78)
    print()
    print("(API 키 없음 — 충실도 프롬프트 구조만 확인)")

print()
print("[충실도가 중요한 이유]")
print("  28장 5절에서 확인했듯, 검색이 맞아도 LLM 이 지어낼 수 있다.")
print("  충실도는 그것을 잡아내는 지표다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("RAG 평가 종합 — 어디가 문제인가")
print("=" * 78)
print()
print("두 지표를 조합하면 문제의 위치를 알 수 있다.")
print()
print(f"{'검색 재현율':<16}{'충실도':<16}{'진단':<26}{'대응'}")
print("-" * 78)
diagnoses = [
    ("높음", "높음", "정상",                "—"),
    ("높음", "낮음", "**LLM 이 지어냄**",    "프롬프트 강화 (28장 6절)"),
    ("낮음", "높음", "**검색 실패**",        "29장의 개선 기법"),
    ("낮음", "낮음", "둘 다 문제",           "검색부터 고친다"),
]
for a, b, c, d in diagnoses:
    print(f"{a:<16}{b:<16}{c:<26}{d}")
print("-" * 78)
print()

fig, ax = plt.subplots(figsize=(7, 5.5))

# 사분면 그리기
ax.axhline(0.5, color="gray", linewidth=1.5)
ax.axvline(0.5, color="gray", linewidth=1.5)

quadrants = [
    (0.75, 0.75, "정상", "#0D9488"),
    (0.75, 0.25, "LLM이\n지어냄", "#EA580C"),
    (0.25, 0.75, "검색\n실패", "#DC2626"),
    (0.25, 0.25, "둘 다\n문제", "#7C1D1D"),
]
for x, y, label, color in quadrants:
    ax.add_patch(plt.Rectangle((x - 0.25, y - 0.25), 0.5, 0.5,
                                facecolor=color, alpha=0.18))
    ax.text(x, y, label, ha="center", va="center",
            fontsize=12, color=color, fontweight="bold")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("검색 재현율 →")
ax.set_ylabel("충실도 →")
ax.set_title("RAG 문제 진단 지도")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("[사용법]")
print("  실패한 질문들을 이 지도 위에 찍어 본다.")
print("  어느 사분면에 몰려 있는지 보면 무엇부터 고칠지 알 수 있다.")

---

## 6. 평가 데이터 만들기 — 이론편 21.5절

**평가의 출발점은 데이터다.** 32장 4절에서도 강조했다.

**직접 만들기 vs LLM으로 만들기** — 각각 장단이 있다.

In [ ]:
print("=" * 78)
print("평가 데이터 확보 방법")
print("=" * 78)
print()
print(f"{'방법':<24}{'품질':<12}{'비용':<12}{'적합한 경우'}")
print("-" * 78)
methods = [
    ("사용자 로그에서 추출", "높음",   "중간",  "서비스 운영 중"),
    ("도메인 전문가 작성",   "매우 높음", "높음",  "정확도가 중요"),
    ("LLM 으로 생성",       "중간",   "낮음",  "빠른 시작"),
    ("공개 벤치마크",       "중간",   "없음",  "일반 능력 비교"),
]
for a, b, c, d in methods:
    print(f"{a:<24}{b:<12}{c:<12}{d}")
print("-" * 78)
print()

GENERATE_EVAL = """주어진 문서를 읽고 평가용 질문과 정답을 만드세요.

조건:
- 문서에서 답을 찾을 수 있는 질문만 만듭니다.
- 난이도를 섞습니다 (직접 언급된 것 / 추론이 필요한 것).
- 실제 사용자가 쓸 법한 표현으로 씁니다.

아래 JSON 형식으로만 답하세요.
{"items": [{"question": "질문", "answer": "정답", "difficulty": "쉬움/보통/어려움"}]}"""


def generate_eval_data(document, n=3):
    """문서에서 평가 데이터를 만든다"""
    user = f"[문서]\n{document}\n\n{n}개의 질문-정답 쌍을 만드세요."
    result = call_llm([
        {"role": "system", "content": GENERATE_EVAL},
        {"role": "user", "content": user},
    ], max_tokens=600)

    if result is None:
        return None
    cleaned = result.strip().replace("```json", "").replace("```", "")
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start:end+1]
    try:
        return json.loads(cleaned).get("items", [])
    except json.JSONDecodeError:
        return None


sample_doc = ("교육비는 연간 200만원까지 지원되며 업무 관련성이 인정되어야 합니다. "
              "수료증 제출이 필요하고 어학 교육은 100만원까지입니다. "
              "수료하지 못한 경우 지원금을 반납해야 합니다.")

print("LLM 으로 평가 데이터 생성")
print(f"문서: {sample_doc[:50]}...")
print()

if API_KEY:
    items = generate_eval_data(sample_doc, 4)
    if items:
        print(f"{'질문':<38}{'정답':<26}{'난이도'}")
        print("-" * 78)
        for item in items:
            print(f"{item.get('question','')[:36]:<38}"
                  f"{item.get('answer','')[:24]:<26}"
                  f"{item.get('difficulty','')}")
        print("-" * 78)
else:
    print("(API 키 없음 — 생성 프롬프트 구조)")
    print("-" * 78)
    print(GENERATE_EVAL)
    print("-" * 78)

print()
print("[LLM 생성 데이터의 주의점]")
print("  1) 문서를 그대로 베낀 질문이 나오기 쉽다 — 너무 쉬움")
print("  2) 사람이 실제로 묻는 방식과 다를 수 있다")
print("  3) **반드시 사람이 검토**해야 한다")
print()
print("  29장 1절에서 '어려운 질문'을 쓴 이유가 이것이다.")
print("  쉬운 질문만 있으면 개선 여지가 보이지 않는다.")

In [ ]:
import numpy as np

print("=" * 78)
print("좋은 평가 데이터의 조건")
print("=" * 78)
print()
print(f"{'조건':<24}{'이유':<34}{'확인 방법'}")
print("-" * 78)
conditions = [
    ("난이도가 섞여 있음",   "쉬운 것만 있으면 차이가 안 보임", "기준 성능이 60~80%"),
    ("실제 사용 표현",      "실험실 성능 ≠ 실제 성능",       "사용자 로그와 대조"),
    ("충분한 수",          "적으면 우연에 좌우됨",           "최소 50, 권장 200+"),
    ("정답이 명확",        "채점이 흔들리지 않게",           "두 사람이 같게 판단"),
    ("최신 상태",          "문서가 바뀌면 정답도 바뀜",       "정기 점검"),
]
for a, b, c in conditions:
    print(f"{a:<24}{b:<34}{c}")
print("-" * 78)
print()

print("[표본 수와 신뢰 구간]")
print()
print("  정확도 p 를 n 개 표본으로 측정했을 때의 오차 범위")
print()
print(f"{'표본 수':<14}{'p=0.5 일 때 95% 구간':<28}{'구별 가능한 차이'}")
print("-" * 78)
for n in [20, 50, 100, 200, 500, 1000]:
    p = 0.5
    se = np.sqrt(p * (1 - p) / n)
    margin = 1.96 * se
    print(f"{n:<14}±{margin*100:>5.1f}%p                      "
          f"{margin*2*100:>5.1f}%p 이상")
print("-" * 78)
print()
print("  표본이 50개면 오차가 ±14%p 다.")
print("  즉 '70% vs 78%' 정도의 차이는 구별할 수 없다.")
print()
print("  → 개선 효과를 확인하려면 최소 200개는 있어야 한다")

---

## 7. 회귀 시험 — 이론편 25.5절

**"어제는 됐는데 오늘 안 된다"를 막는다.**

41장에서 다룬 버전 관리와 짝을 이룬다.
무언가를 바꿀 때마다 **이전에 되던 것이 여전히 되는지** 확인한다.

In [ ]:
import json
from pathlib import Path

print("=" * 78)
print("회귀 시험 구조")
print("=" * 78)


class RegressionSuite:
    """회귀 시험 모음 — 반드시 통과해야 하는 항목들"""

    def __init__(self):
        self.cases = []

    def add(self, name, question, check_fn, category="일반"):
        """check_fn(answer) → True/False"""
        self.cases.append({
            "name": name, "question": question,
            "check": check_fn, "category": category,
        })

    def run(self, answer_fn, verbose=True):
        results = []
        for case in self.cases:
            try:
                answer = answer_fn(case["question"])
                passed = case["check"](answer)
                error = None
            except Exception as e:
                answer, passed, error = None, False, str(e)[:60]

            results.append({
                "name": case["name"], "category": case["category"],
                "passed": passed, "answer": answer, "error": error,
            })

            if verbose:
                mark = "통과" if passed else "실패"
                print(f"  [{mark}] {case['name']:<34}{str(answer)[:26]}")

        return results


# 시험 항목 정의
suite = RegressionSuite()

suite.add("수도 질문", "대한민국의 수도는?",
          lambda a: a and "서울" in a, "사실")
suite.add("숫자 계산", "12 곱하기 7은?",
          lambda a: a and "84" in a.replace(",", ""), "계산")
suite.add("모르는 것 거부", "2099년 월드컵 우승국은?",
          lambda a: a and any(k in a for k in ["모르", "없", "알 수 없", "예측"]),
          "안전")
suite.add("형식 준수", "1부터 3까지 숫자만 쉼표로",
          lambda a: a and all(str(i) in a for i in [1, 2, 3]), "형식")

print(f"등록된 시험: {len(suite.cases)}건")
print()
print(f"{'항목':<24}{'분류':<12}{'검증 방법'}")
print("-" * 78)
for c in suite.cases:
    print(f"{c['name']:<24}{c['category']:<12}함수로 자동 판정")
print("-" * 78)
print()


# 가상의 답변 함수로 시험해 본다
def mock_answer(question):
    """실제로는 여기에 LLM 이나 RAG 시스템이 들어간다"""
    if "수도" in question:
        return "대한민국의 수도는 서울입니다."
    if "곱하기" in question:
        return "12 × 7 = 84 입니다."
    if "2099" in question:
        return "미래의 일이므로 알 수 없습니다."
    if "숫자만" in question:
        return "1, 2, 3"
    return "답변을 생성할 수 없습니다."


print("시험 실행")
results = suite.run(mock_answer)
print()
n_pass = sum(1 for r in results if r["passed"])
print(f"결과: {n_pass}/{len(results)} 통과")

In [ ]:
import json
from pathlib import Path

print("=" * 78)
print("배포 전 자동 점검")
print("=" * 78)
print()

code_example = [
    "# CI 파이프라인에 넣는다 (GitHub Actions 등)",
    "",
    "def test_regression():",
    "    suite = load_regression_suite('tests/cases.json')",
    "    results = suite.run(production_answer_fn)",
    "",
    "    failures = [r for r in results if not r['passed']]",
    "",
    "    # 안전 항목은 하나라도 실패하면 배포 중단",
    "    critical = [f for f in failures if f['category'] == '안전']",
    "    assert not critical, f'안전 시험 실패: {critical}'",
    "",
    "    # 전체 통과율이 기준 미만이면 경고",
    "    pass_rate = 1 - len(failures) / len(results)",
    "    assert pass_rate >= 0.95, f'통과율 {pass_rate:.1%} < 95%'",
]
for line in code_example:
    print("  " + line)

print()
print("-" * 78)
print("[분류별 기준을 다르게]")
print()
print(f"{'분류':<16}{'기준':<24}{'실패 시'}")
print("-" * 78)
print(f"{'안전':<16}{'100% 통과 필수':<24}배포 중단")
print(f"{'사실':<16}{'95% 이상':<24}배포 중단")
print(f"{'형식':<16}{'90% 이상':<24}경고")
print(f"{'품질':<16}{'기준선 대비 하락 없음':<24}검토")
print("-" * 78)
print()
print("[언제 돌리나]")
print("  - 프롬프트를 바꿨을 때")
print("  - 모델을 바꿨을 때 (버전 업그레이드 포함)")
print("  - RAG 문서를 갱신했을 때")
print("  - 정기적으로 (모델 API 는 예고 없이 바뀔 수 있다)")
print()
print("41장의 '재현 가능성'과 함께 보면 좋다.")

---

## 8. 평가 파이프라인 — 이론편 25.5절

지금까지의 것을 **하나로 묶는다.**

In [ ]:
import numpy as np
import json
import time


class EvaluationPipeline:
    """평가 파이프라인 — 여러 지표를 함께 계산한다"""

    def __init__(self, use_llm_judge=False):
        self.use_llm_judge = use_llm_judge and (API_KEY is not None)
        self.results = []

    def evaluate_one(self, question, prediction, reference, context=None):
        """한 건을 여러 지표로 평가"""
        record = {
            "question": question,
            "prediction": prediction,
            "reference": reference,
            "em": int(exact_match(prediction, reference)),
            "f1": token_f1(prediction, reference),
            "semantic": semantic_similarity(prediction, reference),
        }

        if self.use_llm_judge:
            score, reason = judge_binary(question, prediction, reference)
            record["llm_score"] = score
            record["llm_reason"] = reason

        self.results.append(record)
        return record

    def summary(self):
        """전체 요약"""
        if not self.results:
            return {}

        summary = {
            "n": len(self.results),
            "em": np.mean([r["em"] for r in self.results]),
            "f1": np.mean([r["f1"] for r in self.results]),
            "semantic": np.mean([r["semantic"] for r in self.results]),
        }
        llm_scores = [r.get("llm_score") for r in self.results
                      if r.get("llm_score") is not None]
        if llm_scores:
            summary["llm_score"] = np.mean(llm_scores)
        return summary

    def failures(self, threshold=0.5):
        """낮은 점수 사례 — 개선 대상"""
        key = "llm_score" if self.use_llm_judge else "semantic"
        return [r for r in self.results
                if (r.get(key) or 0) < threshold]


# 평가 실행
eval_cases = [
    ("대한민국의 수도는?", "서울입니다.", "서울"),
    ("1년은 몇 개월?", "12개월입니다.", "12개월"),
    ("물의 화학식은?", "H2O 입니다.", "H2O"),
    ("가장 큰 대륙은?", "유럽입니다.", "아시아"),
    ("빛의 속도는?", "약 30만 km/s 입니다.", "초속 약 30만 킬로미터"),
]

print("=" * 78)
print("평가 파이프라인 실행")
print("=" * 78)

pipeline = EvaluationPipeline(use_llm_judge=True)

t0 = time.time()
for q, pred, ref in eval_cases:
    pipeline.evaluate_one(q, pred, ref)
elapsed = time.time() - t0

print(f"{len(eval_cases)}건 평가: {elapsed:.1f}초")
print()

header = f"{'질문':<22}{'EM':<8}{'F1':<10}{'의미':<10}"
if pipeline.use_llm_judge:
    header += f"{'LLM':<8}"
print(header)
print("-" * 78)
for r in pipeline.results:
    row = (f"{r['question'][:20]:<22}{r['em']:<8}"
           f"{r['f1']:<10.3f}{r['semantic']:<10.3f}")
    if pipeline.use_llm_judge:
        row += f"{str(r.get('llm_score')):<8}"
    print(row)
print("-" * 78)

s = pipeline.summary()
print()
print("평균")
for k, v in s.items():
    if k != "n":
        print(f"  {k:<12}{v:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 지표별 분포 ---
ax = axes[0]
metrics_data = {
    "EM": [r["em"] for r in pipeline.results],
    "F1": [r["f1"] for r in pipeline.results],
    "의미 유사도": [r["semantic"] for r in pipeline.results],
}
if pipeline.use_llm_judge:
    llm_vals = [r.get("llm_score") for r in pipeline.results
                if r.get("llm_score") is not None]
    if llm_vals:
        metrics_data["LLM Judge"] = llm_vals

x = np.arange(len(metrics_data))
means = [np.mean(v) for v in metrics_data.values()]
colors_m = ["#94A3B8", "#EA580C", "#1E40AF", "#0D9488"][:len(means)]

bars = ax.bar(x, means, color=colors_m)
for b, m in zip(bars, means):
    ax.text(b.get_x() + b.get_width()/2, m + 0.02, f"{m:.3f}",
            ha="center", fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(list(metrics_data.keys()), fontsize=9)
ax.set_ylabel("평균 점수")
ax.set_ylim(0, 1.15)
ax.set_title("지표별 평균")
ax.grid(axis="y", alpha=0.3)

# --- 오른쪽: 사례별 비교 ---
ax = axes[1]
n_cases = len(pipeline.results)
idx = np.arange(n_cases)
w = 0.25
ax.bar(idx - w, [r["em"] for r in pipeline.results], w,
       label="EM", color="#94A3B8")
ax.bar(idx, [r["f1"] for r in pipeline.results], w,
       label="F1", color="#EA580C")
ax.bar(idx + w, [r["semantic"] for r in pipeline.results], w,
       label="의미", color="#1E40AF")
ax.set_xticks(idx)
ax.set_xticklabels([f"{i+1}" for i in idx])
ax.set_xlabel("사례 번호")
ax.set_ylabel("점수")
ax.set_title("사례별 지표 차이")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 78)
print("지표가 서로 다르게 나오는 것이 정상이다")
print("=" * 78)
print()
print("  EM 은 대부분 0 이다 — 조사·어미 때문")
print("  의미 유사도는 대체로 높다 — 오답도 높게 나올 수 있음")
print("  LLM Judge 가 사람 판단에 가장 가깝다 (4절에서 검증 필요)")
print()
print("[실무의 조합]")
print("  1차: 문자열·의미 지표로 빠르게 걸러낸다 (비용 없음)")
print("  2차: 애매한 것만 LLM Judge 로 정밀 평가 (비용 있음)")
print("  3차: 표본을 사람이 검증 (카파 확인)")

---

## 9. 정리

### 지표별 특성

| 지표 | 잡는 것 | 놓치는 것 | 비용 |
|---|---|---|---|
| EM | 정확 일치 | **표현 차이** | 없음 |
| F1 | 토큰 겹침 | 순서·의미 | 없음 |
| ROUGE | n-gram 겹침 | 재구성된 표현 | 없음 |
| 의미 유사도 | 표현 차이 | **오답 (부산≈서울)** | 낮음 |
| **LLM Judge** | 대부분 | 편향 있음 | **호출 비용** |

### 기억할 것

| 항목 | 요점 |
|---|---|
| EM/F1 | 한국어는 조사·어미 때문에 특히 낮게 나옴 |
| 의미 유사도 | **오답을 못 거른다** (같은 범주면 높음) |
| LLM Judge | 위치·길이·자기선호 편향 주의 |
| 편향 대응 | 양방향 평가, 다수결, 다른 모델로 채점 |
| **카파** | 우연 일치를 뺀 실제 일치도 — 0.6 이상 |
| RAG 평가 | 검색과 생성을 **따로** |
| 표본 수 | 50개면 ±14%p — 최소 200개 |
| 회귀 시험 | 안전 항목은 100% 통과 필수 |

### 다른 장들이 남긴 숙제에 대한 답

| 장 | 숙제 | 이 장의 답 |
|---|---|---|
| 33장 | "퍼플렉서티만으로 부족" | 과제 기반 평가 + LLM Judge |
| 32장 | "품질 측정이 어렵다" | 3~4절 |
| 28장 | "답변이 맞는지 어떻게 아나" | 5절 충실도 |

### 다음 장

**40. 배포 — 만든 것을 서비스로** — 만든 것을 실제로 운영한다.
이 장에서 만든 평가가 **운영 지표의 한 축**이 된다.